In [1]:
from pynq import Overlay
from pynq import MMIO
import numpy as np
import time
import struct

import cv2

In [42]:
def reshape_input_and_weights(x, weights, fc_key='fc', kernel_size=3, oc_size=10):
    # Flatten input
    input_flat = x.flatten()
    
    # Calculate padding length
    total_length = np.ceil(len(input_flat) / (kernel_size * kernel_size * 8)) * (kernel_size * kernel_size * 8)
    pad_length = int(total_length - len(input_flat))
    
    # Pad input
    input_padded = np.pad(input_flat, (0, pad_length), mode='constant', constant_values=0)
    
    # Reshape input
    ic = int(total_length // (kernel_size * kernel_size))
    input_reshaped = input_padded.reshape(ic, kernel_size, kernel_size)
    
    # Load weight data
    weight_data = weights[fc_key]
    
    # Initialize reshaped weights array
    weight_reshaped = np.zeros((oc_size, ic, kernel_size, kernel_size), dtype=weight_data.dtype)
    
    # Reshape weights for each output channel
    for oc in range(oc_size):
        weight_flat = weight_data[oc]
        weight_padded = np.pad(weight_flat, (0, pad_length), mode='constant', constant_values=0)
        weight_oc_reshaped = weight_padded.reshape(ic, kernel_size, kernel_size)
        weight_reshaped[oc] = weight_oc_reshaped
    
    return input_reshaped, weight_reshaped

In [43]:
def quantize_8bit(x):
    # PyTorch tensor를 numpy로 변환
    temp = x
    temp = np.floor(temp)  # Round the values ######################################################
    # 9 LSB 제거 (오른쪽 시프트 9)
    output_shifted = np.right_shift(temp.astype(np.int32), 9)
    # 8-bit signed int 범위로 클리핑 (-128 ~ 127)
    quantized = np.clip(output_shifted, -128, 127).astype(np.int8)
    return quantized

In [52]:
class NPUDriver:
    def __init__(self, bitfile_path):
        self.hw = Overlay(bitfile_path)
        self.csr = self.hw.csr_0.mmio.array        #         self.csr = self.hw.csr_0
        self.imem = self.hw.INPUT_MEM.mmio.array
        self.omem = self.hw.OUT_MEM.mmio.array
        self.wmem = self.hw.WEIGHT_MEM.mmio.array
        print(f"IMEM Shape: {self.imem.shape}")
        print(f"WMEM Shape: {self.wmem.shape}")
        print(f"OMEM Shape: {self.omem.shape}")

    def write_csr(self, address, value):
        """CSR write"""
        self.csr[address//4] = value                              # self.csr.write(address, value) 

    def read_csr(self, address):
        return self.csr[address//4]                            # self.csr.write(address, value) 

    def config_layer(self, kw , kh, ic, input_w, input_h, oc):
        """레이어 config 설정"""
        # 공통 config
        self.write_csr(0x08, kw)
        self.write_csr(0x0C, kh)
        self.write_csr(0x10, ic)
        self.write_csr(0x14, input_w)
        self.write_csr(0x18, input_h)
        self.write_csr(0x1C, oc)

    def load_data(self, input_data, weight_data):
        """데이터 로드: WMEM (weight), IMEM (input) -> flatten 시켜서"""
        TOTAL_SIZE = 16384
    
        input_data_size = input_data.size
        weight_data_size = weight_data.size
        
        in_data_float32 = input_data.astype(np.float32).ravel()
        weight_data_float32 = weight_data.astype(np.float32).ravel()
        
        # float32를 16진수 정수로 변환
        in_hex_int = [struct.unpack('<I', np.float32(x).tobytes())[0] for x in in_data_float32]
        weight_hex_int = [struct.unpack('<I', np.float32(x).tobytes())[0] for x in weight_data_float32]
        
        # input 데이터 패딩
        padded_in_hex_int = in_hex_int[:input_data_size] + [0] * (TOTAL_SIZE - input_data_size)
        self.imem[0:TOTAL_SIZE] = padded_in_hex_int

        # weight 데이터 패딩
        padded_weight_hex_int = weight_hex_int[:weight_data_size] + [0] * (TOTAL_SIZE - weight_data_size)
        self.wmem[0:TOTAL_SIZE] = padded_weight_hex_int

    def get_data(self, num_elements):
        """데이터 가져오기: OMEM (output)"""
        return self.omem[0:num_elements]

    def start_npu(self, input_data, weight_data, kw , kh, ic, input_w, input_h, oc):
        """데이터 로드: WMEM (weight), IMEM (input)"""
        self.load_data(input_data, weight_data)
        """레이어 config 설정"""
        self.config_layer(kw , kh, ic, input_w, input_h, oc)
        """Start Signal pulse"""
        self.write_csr(0x04, 1)
        print(" ")
        print("NPU Started")

        
        while True:
            if (self.read_csr(0x00) == 1): # self.csr[0] == 1
                break
        return self.get_data((input_w-2)*(input_h-2)*oc)
    
    def run_conv_2d(self, input_data, weight_data, tile_h=8, tile_w=8, tile_oc=8):
        """
        conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        stride = 1
        input_ch, input_h, input_w = input_data.shape
        output_ch, _, kernel_h, kernel_w = weight_data.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w)).astype(np.float32)

        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1
                    
                    i_act_tile = input_data[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight_data[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range)
                    tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape
                    
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    tile_oc_act_flat = tile_oc_act_flat.view(np.float32)
#                     tile_oc_act_flat = quantize_8bit(tile_oc_act_flat)
                    ########################################################################################################################
                    # convolution 연산
#                     tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)
                    
                    o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act

        return o_act

    def run_fc_2d(self, i_act, weight, stride=1, tile_h=3, tile_w=3, tile_oc=1, padding=0):
        """
        Implement FC with conv2d (9 for loops) with tiling.
        - i_act: shape (input_ch, input_h, input_w)
        - weight: shape (output_ch, input_ch, kernel_h, kernel_w)
        """
        input_ch, input_h, input_w = i_act.shape
        output_ch, _, kernel_h, kernel_w = weight.shape

        # 출력 feature map 크기 계산
        output_h = (input_h - kernel_h) // stride + 1
        output_w = (input_w - kernel_w) // stride + 1
        o_act = np.zeros((output_ch, output_h, output_w))
        
        out_addr = 0
        
        # 타일 단위 반복
        for oh in range(0, output_h, tile_h):
            for ow in range(0, output_w, tile_w):
                for oc in range(0, output_ch, tile_oc):
                    h_range = min(tile_h, output_h - oh)
                    w_range = min(tile_w, output_w - ow)
                    oc_range = min(tile_oc, output_ch - oc)

                    # 타일 단위로 i_act와 weight 슬라이싱
                    h_start = oh * stride
                    w_start = ow * stride
                        
                    h_end = h_start + h_range * stride + kernel_h - 1
                    w_end = w_start + w_range * stride + kernel_w - 1

                    i_act_tile = i_act[:, h_start:h_end, w_start:w_end]
                    weight_tile = weight[oc:oc + oc_range, :, :, :]

                    tile_oc_act_flat = np.zeros(oc_range * h_range * w_range)
                    tile_o_act = np.zeros((oc_range, h_range, w_range)).astype(np.float32)
                    
                    input_ch, tile_h_input, tile_w_input = i_act_tile.shape
                    oc_range, input_ch_w, kernel_h, kernel_w = weight_tile.shape
                    oc_range_out, h_range, w_range = tile_o_act.shape
                    
                    ########################################################################################################################
                    # convolution 연산
                    tile_oc_act_flat = self.start_npu(i_act_tile, weight_tile, kernel_w, kernel_h, input_ch, tile_w_input, tile_h_input, oc_range)
                    tile_oc_act_flat = tile_oc_act_flat.view(np.float32)
                    tile_oc_act_flat = quantize_8bit(tile_oc_act_flat)
                    
                    ########################################################################################################################
                    # convolution 연산
#                     tile_oc_act_flat = conv2d_IN_HW(i_act_tile, weight_tile, tile_oc_act_flat, quantize_8bit, kernel_h, kernel_w, input_ch, oc_range, h_range, w_range)
                    ########################################################################################################################
                    tile_o_act = tile_oc_act_flat.reshape(oc_range, h_range, w_range)

#                     o_act[oc:oc + oc_range, oh:oh + h_range, ow:ow + w_range] = tile_o_act
                    o_act[out_addr] = tile_o_act

                    out_addr += 1

        return o_act


In [53]:
driver = NPUDriver("no_quant_1.bit")

IMEM Shape: (131072,)
WMEM Shape: (131072,)
OMEM Shape: (131072,)


In [54]:
input_image = np.load('npy_files/input.npy')
weights = {
    'conv1': np.load('npy_files/layer1_0_weight.npy'),
    'conv2': np.load('npy_files/layer2_0_weight.npy'),
    'conv3': np.load('npy_files/layer3_0_weight.npy'),
    'conv4': np.load('npy_files/layer4_0_weight.npy'),
    'fc': np.load('npy_files/fc1_weight.npy')
}

In [56]:

####################### Test With 2 different Tiles ###########################
input_data = np.load('i_act_tile_2.npy')
weight = np.load('weight_tile_2.npy')
# output = driver.start_npu(input_data, weight, 3 , 3, 64, 6, 10, 8)
output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
print(output)
print(type(output))
print(output.shape)
result = quantize_8bit(output)
print(result)
###############################################################################

########################### Test CONV1 Layer ###############################
# input_image = np.load('npy_files/input.npy')
# input_data = input_image[0]
# weight = np.load('npy_files/layer1_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv1_output.txt', output.flatten(), fmt='%f')
###############################################################################

########################### Test CONV2 Layer ###############################
# input_data = np.load('npy_files/conv1_n_leaky_output.npy')
# weight = np.load('npy_files/layer2_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv2_output.txt', output.flatten(), fmt='%f')        # .reshape(-1, output.shape[-1])
###############################################################################

########################### Test CONV3 Layer ###############################
# input_data = np.load('npy_files/conv2_n_leaky_output.npy')
# weight = np.load('npy_files/layer3_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv3_output.txt', output.flatten(), fmt='%f')
###############################################################################

########################### Test CONV4 Layer ###############################
# input_data = np.load('npy_files/conv3_n_leaky_output.npy')
# weight = np.load('npy_files/layer4_0_weight.npy')
# output = driver.run_conv_2d(input_data, weight, tile_h=8, tile_w=8, tile_oc=8)
# np.savetxt('conv4_output.txt', output.flatten(), fmt='%f')                         # .reshape(-1, output.shape[-1])
###############################################################################

########################### Test FC Layer ###############################
# x = np.load('npy_files/conv4_n_maxpool_output.npy')
# weights_fc = {'fc': np.load('npy_files/fc1_weight.npy')}

# input_reshaped, weight_reshaped = reshape_input_and_weights(x, weights_fc)

# output = driver.run_fc_2d(input_reshaped, weight_reshaped, tile_h=3, tile_w=3, tile_oc=1).reshape(-1)
# print(output)
###############################################################################

 
NPU Started
[[[-6.103250e+03 -5.211500e+03 -2.954000e+03 -9.112500e+02]
  [-1.123000e+03 -3.001500e+03 -1.229750e+03 -4.245000e+02]
  [-5.266000e+03 -4.176500e+03 -1.161500e+03  5.550000e+02]
  [-2.744500e+03 -8.780000e+02  1.387500e+02  1.775000e+01]
  [-3.302250e+03 -8.327500e+02 -6.400000e+01  2.110000e+02]
  [-1.690000e+03 -3.625000e+01  9.775000e+01  1.555000e+02]
  [-8.227500e+02  3.925000e+01  2.645000e+02  8.650000e+01]
  [ 1.275000e+01 -2.025000e+01  1.627500e+02 -7.500000e-01]]

 [[-5.776750e+03 -5.647500e+02  5.725000e+02 -1.890000e+02]
  [-4.916750e+03  1.903750e+03 -9.710000e+02 -5.377500e+02]
  [-2.310500e+03 -5.205000e+02  1.625000e+02 -7.807500e+02]
  [ 5.842500e+02 -1.642000e+03 -1.344250e+03  4.875000e+01]
  [ 6.370000e+02 -3.105000e+02 -1.161250e+03  2.387500e+02]
  [-1.379000e+03 -1.048000e+03 -2.100000e+01  2.047500e+02]
  [-8.015000e+02 -8.175000e+02  3.227500e+02  8.100000e+01]
  [-9.447500e+02  2.925000e+01  1.985000e+02  1.162500e+02]]

 [[-6.586000e+03 -1.51

In [33]:
def int32_array_to_float32_array(int_array):
#     # 입력 배열을 uint32 타입으로 변환 (필요한 경우)
#     int_array = np.array(int_array, dtype=np.uint32)
#     # int32 배열을 float32로 재해석
    float_array = int_array.view(np.float32)
    return float_array

In [34]:
result = int32_array_to_float32_array(output)
quantize_8bit(result)
# print(f"입력 int32 배열: {output}, 변환된 float32 배열: {result}")

array([-12, -11,  -6,  -2,  -3,  -6,  -3,  -1, -11,  -9,  -3,   1,  -6,
        -2,   0,   0,  -7,  -2,  -1,   0,  -4,  -1,   0,   0,  -2,   0,
         0,   0,   0,  -1,   0,  -1, -12,  -2,   1,  -1, -10,   3,  -2,
        -2,  -5,  -2,   0,  -2,   1,  -4,  -3,   0,   1,  -1,  -3,   0,
        -3,  -3,  -1,   0,  -2,  -2,   0,   0,  -2,   0,   0,   0, -13,
        -3,  -4,  -3,  -5, -10,  -2,  -1,  -2,  -2,  -2,  -1, -10,  -4,
        -1,   0,  -4,  -2,   0,  -1,  -3,  -1,  -1,  -1,  -2,  -1,  -1,
         0,   0,  -1,   0,   0, -15,   5,  -9,  -3,  -9,  -3,  -5,  -2,
         5,  -7,  -4,   1,  -4,  -3,  -1,   0,  -7,  -2,   1,   0,  -4,
         0,   0,   0,  -2,   2,   0,   0,   0,   1,   0,   0, -11, -10,
        -8,  -3, -20,  -3,   1,  -1, -10,  -4,  -4,   0,  -6,   2,   0,
         0,  -3,  -1,   1,  -1,   0,   0,  -1,  -1,  -1,   0,  -1,  -1,
         0,  -1,  -1,   0,  -4, -10,  -4,  -6,  -4, -12,  -4,  -3, -12,
        -3,  -4,  -2, -15,  -7,  -2,  -1,  -3,  -5,  -2,  -1,  -